# Aggregate Daily Features to Monthly Resolution

**Purpose:** Collapse the completed daily feature table into a monthly panel.

**Aggregation methods applied:**

| Variable(s) | Method | Used in model? |
|---|---|---|
| `SWE`, `dead_fuel_moisture_1000hr/100hr`, `max/min_air_temperature`, `max/min_relative_humidity`, `specific_humidity`, `surface_downwelling_shortwave_flux_in_air`, `LAI` | mean, min, max | mean ✓ |
| `precipitation_amount` | mean, min, max | mean ✓ — dry days zero-filled, so mean = total ÷ days-in-month |
| `population_density` | mean only | ✓ — constant within year; min/max dropped |
| `wind_speed` | **mean** (arithmetic) | ✓ — used directly as `wind_speed` |
| `wind_direction_category` | **Mode** of daily compass octant | ✓ — one-hot encoded in model pipeline |
| `wind_direction_category_*` octant counts | Count of days per octant | available as `*_freq` columns, not in final model |
| `wind_speed_uv`, `wind_direction_uv` | Vector decomposition (u/v) | computed for reference, **not used in final model** |

**Pipeline Overview:**

| Phase | Steps |
|-------|-------|
| **A. Standard Aggregation** | mean/min/max for all numeric variables including `wind_speed` |
| **B. Wind Direction Mode & Octant Counts** | Monthly mode of compass octant + day count per octant |
| **C. Wind Vector Aggregation** *(reference only)* | u/v decomposition → `wind_speed_uv`, `wind_direction_uv` — not used in final model |

**Input:** `Clean_Data/Feature_Data/Daily_Features_Completed.parquet`

**Outputs:**
- `Clean_Data/Monthly/weather_features.parquet` — mean/min/max aggregates (includes `wind_speed`)
- `Clean_Data/Monthly/wind_direction_features.parquet` — mode + octant day counts
- `Clean_Data/Monthly/wind_direction_features_nv.parquet` — vector-averaged wind *(reference)*

> **Note:** PDSI and OpenStreetMap density features are added at monthly resolution
> in `03_02_Agg_-_PDSI_OpenStreetMap_to_Monthly.ipynb`.

## 0. Configuration

Centralized path configuration — **edit this cell only**.

In [1]:
import os

PROJECT_ROOT = r"E:\zcao\CA_Wildfire"

INPUT_PATH  = os.path.join(PROJECT_ROOT, "Clean_Data", "Feature_Data",
                           "Daily_Features_Completed.parquet")
MONTHLY_DIR = os.path.join(PROJECT_ROOT, "Clean_Data", "Monthly")

os.makedirs(MONTHLY_DIR, exist_ok=True)

print(f"Input  : {INPUT_PATH}")
print(f"Output : {MONTHLY_DIR}")
print(f"  [{'OK' if os.path.exists(INPUT_PATH) else 'MISSING'}] INPUT_PATH")

Input  : E:\zcao\CA_Wildfire\Clean_Data\Feature_Data\Daily_Features_Completed.parquet
Output : E:\zcao\CA_Wildfire\Clean_Data\Monthly
  [OK] INPUT_PATH


## 1. Environment Setup

In [2]:
import sys, gc, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

gc.collect()
print(f"Python : {sys.version.split('|')[0].strip()}")
print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")

Python : 3.9.13 (main, Aug 25 2022, 23:51:50) [MSC v.1916 64 bit (AMD64)]
pandas : 2.2.2
numpy  : 1.24.4


---

# Phase A: Standard Aggregation (mean / min / max)

`wind_speed` is included here as an arithmetic mean — this is the value
used in the final model. `wind_from_direction` is excluded from this phase
because directional averaging requires the vector method in Phase C.

## 2. Load Daily Features

Load the completed daily table, add a `year_month` period column,
and confirm shape and date range.

In [3]:
features = pd.read_parquet(INPUT_PATH)
features['year_month'] = features['day'].dt.to_period('M')

print(f"Shape      : {features.shape}")
print(f"Date range : {features['day'].min().date()} → {features['day'].max().date()}")
print(f"\nColumn dtypes:")
print(features.dtypes.to_string())

Shape      : (127137467, 20)
Date range : 1994-01-01 → 2020-09-30

Column dtypes:
day                                          datetime64[ns]
lat                                                 float64
lon                                                 float64
SWE                                                 float32
year                                                  int32
dead_fuel_moisture_1000hr                           float64
dead_fuel_moisture_100hr                            float64
max_air_temperature                                 float64
max_relative_humidity                               float64
min_air_temperature                                 float64
min_relative_humidity                               float64
precipitation_amount                                float64
specific_humidity                                   float64
surface_downwelling_shortwave_flux_in_air           float64
wind_from_direction                                 float32
wind_speed        

## 3. Aggregate Weather Variables

Compute **mean, min, max** for each variable grouped by `(year_month, lat, lon)`.

Notes on specific variables:
- **`wind_speed`** — arithmetic mean; this is the value fed to the model
- **`wind_from_direction`** — excluded here; directional averaging is handled in Phase B (mode) and Phase C (vector)
- **`precipitation_amount`** — dry days zero-filled in `02_04`; mean = total ÷ days-in-month
- **`SWE`** — non-snow cells/seasons zero-filled; mean reflects proportion of snowy days × avg depth
- **`LAI`** — cloud-contaminated retrievals zero-filled; ~3% of days affected
- **`population_density`** — constant within year; only mean kept, min/max dropped

In [4]:
# wind_from_direction is excluded — directional averaging handled in Phase B/C
# wind_speed uses arithmetic mean — this is the value used in the final model
NUM_COLS = [
    'SWE',
    'dead_fuel_moisture_1000hr',
    'dead_fuel_moisture_100hr',
    'max_air_temperature',
    'max_relative_humidity',
    'min_air_temperature',
    'min_relative_humidity',
    'precipitation_amount',
    'specific_humidity',
    'surface_downwelling_shortwave_flux_in_air',
    'wind_speed',          # arithmetic mean — used directly in model
    'population_density',
    'LAI',
]

agg_dict = {}
for col in NUM_COLS:
    agg_dict[col]          = (col, 'mean')
    agg_dict[col + '_min'] = (col, 'min')
    agg_dict[col + '_max'] = (col, 'max')

weather_agg = features.groupby(['year_month','lat','lon']).agg(**agg_dict).reset_index()

# population_density is constant within a year — min/max identical to mean, drop them
drop_pop = [c for c in weather_agg.columns
            if c.startswith('population_density') and c != 'population_density']
weather_agg = weather_agg.drop(columns=drop_pop)

print(f"Aggregated shape : {weather_agg.shape}")
print(f"Columns          : {list(weather_agg.columns)}")

Aggregated shape : (4181379, 40)
Columns          : ['year_month', 'lat', 'lon', 'SWE', 'SWE_min', 'SWE_max', 'dead_fuel_moisture_1000hr', 'dead_fuel_moisture_1000hr_min', 'dead_fuel_moisture_1000hr_max', 'dead_fuel_moisture_100hr', 'dead_fuel_moisture_100hr_min', 'dead_fuel_moisture_100hr_max', 'max_air_temperature', 'max_air_temperature_min', 'max_air_temperature_max', 'max_relative_humidity', 'max_relative_humidity_min', 'max_relative_humidity_max', 'min_air_temperature', 'min_air_temperature_min', 'min_air_temperature_max', 'min_relative_humidity', 'min_relative_humidity_min', 'min_relative_humidity_max', 'precipitation_amount', 'precipitation_amount_min', 'precipitation_amount_max', 'specific_humidity', 'specific_humidity_min', 'specific_humidity_max', 'surface_downwelling_shortwave_flux_in_air', 'surface_downwelling_shortwave_flux_in_air_min', 'surface_downwelling_shortwave_flux_in_air_max', 'wind_speed', 'wind_speed_min', 'wind_speed_max', 'population_density', 'LAI', 'LAI_min',

In [5]:
save_path = os.path.join(MONTHLY_DIR, 'weather_features.parquet')
weather_agg.to_parquet(save_path, index=False)
print(f"Saved -> {save_path}")
print(f"File size : {os.path.getsize(save_path)/1e6:.1f} MB")

del weather_agg; gc.collect()

Saved -> E:\zcao\CA_Wildfire\Clean_Data\Monthly\weather_features.parquet
File size : 328.0 MB


0

---

# Phase B: Wind Direction Mode & Octant Counts

**This is the wind direction method used in the final model.**

- **Mode** — the most frequent compass octant in the month; one-hot encoded
  downstream in the model pipeline (`wind_direction_category_N/NE/…`)
- **Octant day counts** — number of days each of the 8 octants was dominant;
  renamed to `*_freq` columns in the model pipeline (available but not
  in the final feature set)

## 5. Compute Wind Direction Mode & Octant Counts

In [6]:
# --- Octant day counts (one column per compass direction) ---
wind_df = features[['year_month','lat','lon','wind_direction_category']].copy()
wind_dummies = pd.get_dummies(wind_df, columns=['wind_direction_category'])
octant_cols = [c for c in wind_dummies.columns if c.startswith('wind_direction_category_')]

wind_counts = (
    wind_dummies.groupby(['year_month','lat','lon'])[octant_cols]
    .sum().reset_index()
)
wind_counts['wind_direction_days_total'] = wind_counts[octant_cols].sum(axis=1)

# --- Mode of compass octant (used in final model) ---
wind_mode = (
    features.groupby(['year_month','lat','lon'])['wind_direction_category']
    .agg(lambda x: x.mode()[0] if not x.mode().empty else None)
    .reset_index()
)

wind_dir = wind_mode.merge(wind_counts, on=['year_month','lat','lon'], how='inner')
print(f"Wind direction shape : {wind_dir.shape}")
print(f"Columns              : {list(wind_dir.columns)}")
wind_dir.head(3)

Wind direction shape : (4181379, 13)
Columns              : ['year_month', 'lat', 'lon', 'wind_direction_category', 'wind_direction_category_N', 'wind_direction_category_NE', 'wind_direction_category_E', 'wind_direction_category_SE', 'wind_direction_category_S', 'wind_direction_category_SW', 'wind_direction_category_W', 'wind_direction_category_NW', 'wind_direction_days_total']


,year_month,lat,lon,wind_direction_category,wind_direction_category_N,wind_direction_category_NE,wind_direction_category_E,wind_direction_category_SE,wind_direction_category_S,wind_direction_category_SW,wind_direction_category_W,wind_direction_category_NW,wind_direction_days_total
0,1994-01,32.566667,-116.975000,N,12,2,1,0,1,3,2,10,31
1,1994-01,32.566667,-116.933333,N,12,2,1,0,1,3,2,10,31
2,1994-01,32.566667,-116.891667,N,12,2,1,0,1,3,2,10,31


In [7]:
save_path = os.path.join(MONTHLY_DIR, 'wind_direction_features.parquet')
wind_dir.to_parquet(save_path, index=False)
print(f"Saved -> {save_path}")
print(f"File size : {os.path.getsize(save_path)/1e6:.1f} MB")

Saved -> E:\zcao\CA_Wildfire\Clean_Data\Monthly\wind_direction_features.parquet
File size : 12.9 MB


---

# Phase C: Wind Vector Aggregation *(reference — not used in final model)*

Decomposes wind into Cartesian u/v components, averages them monthly,
then recomposes into speed and direction. Handles the 360°/0° wraparound
correctly and weights direction by wind speed.

```
u = -speed × sin(direction°)   # east–west component
v = -speed × cos(direction°)   # north–south component

monthly_u, monthly_v = mean(u), mean(v)
wind_speed_uv     = √(u² + v²)
wind_direction_uv = atan2(-u, -v) × 180/π  (mod 360)
```

> Output is saved for reference. The final model uses `wind_speed` (mean)
> and `wind_direction_category` (mode) from Phases A and B instead.

## 6. Compute Vector-Averaged Wind

In [8]:
features['u'] = -features['wind_speed'] * np.sin(np.deg2rad(features['wind_from_direction']))
features['v'] = -features['wind_speed'] * np.cos(np.deg2rad(features['wind_from_direction']))

agg_uv = features.groupby(['lat','lon','year_month'], as_index=False)[['u','v']].mean()

agg_uv['wind_speed_uv']     = np.sqrt(agg_uv['u']**2 + agg_uv['v']**2)
agg_uv['wind_direction_uv'] = (np.rad2deg(np.arctan2(-agg_uv['u'], -agg_uv['v']))) % 360

wind_nv = agg_uv[['lon','lat','year_month','wind_speed_uv','wind_direction_uv']]
print(f"Vector wind shape : {wind_nv.shape}  (reference only — not used in model)")
wind_nv.head(3)

Vector wind shape : (4181379, 5)  (reference only — not used in model)


,lon,lat,year_month,wind_speed_uv,wind_direction_uv
0,-116.975,32.566667,1994-01,1.716249,332.544003
1,-116.975,32.566667,1994-02,1.294652,241.452107
2,-116.975,32.566667,1994-03,1.984716,282.762726


In [9]:
save_path = os.path.join(MONTHLY_DIR, 'wind_direction_features_nv.parquet')
wind_nv.to_parquet(save_path, index=False)
print(f"Saved -> {save_path}  (reference only)")

# Safe cleanup — only delete variables that exist
for _var in ['features', 'wind_nv', 'wind_dir', 'wind_dummies', 'wind_counts', 'wind_mode']:
    if _var in dir():
        del globals()[_var]
gc.collect()

Saved -> E:\zcao\CA_Wildfire\Clean_Data\Monthly\wind_direction_features_nv.parquet  (reference only)


27

## 6. Summary

| Phase | Output file | Variables | Method | In model? |
|-------|-------------|-----------|--------|----------|
| A | `weather_features.parquet` | SWE, fuel moisture, temperature, humidity, radiation, precipitation, LAI | mean / min / max | mean ✓ |
| A | `weather_features.parquet` | `wind_speed` | arithmetic mean | ✓ |
| A | `weather_features.parquet` | `population_density` | mean only (constant within year) | ✓ |
| B | `wind_direction_features.parquet` | `wind_direction_category` | mode of daily octant → one-hot encoded | ✓ |
| B | `wind_direction_features.parquet` | `wind_direction_category_*` | count of days per octant (as `*_freq`) | available, not in final model |
| C | `wind_direction_features_nv.parquet` | `wind_speed_uv`, `wind_direction_uv` | vector decomposition (u/v) | reference only |